# Day 045 — Exercise 5: run_pipeline

**What you'll build:** `run_pipeline(csv_text: str, session) -> dict` — compose the four steps (extract → validate → transform → load) into a single function that returns a stats dict with keys `'extracted'`, `'loaded'`, and `'skipped'`.

**Why it matters:** A pipeline is a composition of pure functions. Each step has one job: extract returns raw records; validate filters them; transform cleans them; load persists them. `run_pipeline` orchestrates the steps and accumulates counts so the caller can audit what happened without inspecting the database.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import csv
import io
from sqlalchemy import create_engine, String, Float, select, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Sale(Base):
    __tablename__ = 'sales'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    date:     Mapped[str]   = mapped_column(String(20))
    product:  Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    amount:   Mapped[float] = mapped_column()
    region:   Mapped[str]   = mapped_column(String(50))

    def __repr__(self):
        return f'Sale(id={self.id}, product={self.product!r}, amount={self.amount})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


def extract(csv_text: str) -> list:
    reader = csv.DictReader(io.StringIO(csv_text))
    return list(reader)


def validate_record(record: dict) -> bool:
    required = ['date', 'product', 'amount']
    for field in required:
        if not record.get(field, '').strip():
            return False
    try:
        float(record['amount'])
    except (ValueError, TypeError):
        return False
    return True


def transform_record(record: dict) -> dict:
    return {
        'date':     record['date'].strip(),
        'product':  record['product'].strip(),
        'category': record.get('category', '').strip(),
        'amount':   round(float(record['amount']), 2),
        'region':   record.get('region', '').strip().title(),
    }


def load(session, records: list) -> int:
    sales = [Sale(**r) for r in records]
    session.add_all(sales)
    session.commit()
    return len(sales)


CSV_SOURCE = (
    'date,product,category,amount,region\n'
    '2024-01-15,Laptop,Electronics,999.99,East\n'
    '2024-01-16,Headphones,Electronics,149.99,West\n'
    '2024-01-17,Desk Chair,Furniture,349.00,East\n'
    '2024-01-18,,Furniture,199.00,North\n'
    '2024-01-19,Pen Set,Stationery,twelve,South\n'
    '2024-01-20,Monitor,Electronics,599.99,West\n'
    '2024-01-21,Keyboard,Electronics,79.99,East\n'
    '2024-01-22,Webcam,Electronics,,North\n'
    '2024-01-23,Lamp,Furniture,45.99,South\n'
    '2024-01-24,Notebook,Stationery,8.99,West\n'
)

engine  = setup_engine()
session = Session(engine)

## Your Implementation

In [ ]:
def run_pipeline(csv_text: str, session) -> dict:
    """
    Run the full ETL pipeline: extract → validate → transform → load.

    Steps:
    1. raw_records  = extract(csv_text)
    2. valid        = [r for r in raw_records if validate_record(r)]
    3. transformed  = [transform_record(r) for r in valid]
    4. loaded_count = load(session, transformed)
    5. return {'extracted': len(raw_records),
               'loaded':    loaded_count,
               'skipped':   len(raw_records) - loaded_count}
    """
    # TODO: raw_records  = extract(csv_text)
    # TODO: valid        = [r for r in raw_records if validate_record(r)]
    # TODO: transformed  = [transform_record(r) for r in valid]
    # TODO: loaded_count = load(session, transformed)
    # TODO: return {
    # TODO:     'extracted': len(raw_records),
    # TODO:     'loaded':    loaded_count,
    # TODO:     'skipped':   len(raw_records) - loaded_count,
    # TODO: }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'run_pipeline' in globals()
        passed += 1; print('\u2705 Check 1: run_pipeline is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a dict with correct keys
    try:
        stats = run_pipeline(CSV_SOURCE, session)
        assert isinstance(stats, dict), \
            f'expected dict, got {type(stats).__name__}'
        assert {'extracted', 'loaded', 'skipped'} <= set(stats.keys()), \
            f'missing keys: {set(stats.keys())}'
        passed += 1; print(f'\u2705 Check 2: returns dict {stats}')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: extracted == 10 (all CSV rows)
    try:
        assert stats['extracted'] == 10, \
            f'expected extracted=10, got {stats["extracted"]}'
        passed += 1; print(f'\u2705 Check 3: extracted=10')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: loaded == 7, skipped == 3
    try:
        assert stats['loaded'] == 7, \
            f'expected loaded=7, got {stats["loaded"]}'
        assert stats['skipped'] == 3, \
            f'expected skipped=3, got {stats["skipped"]}'
        passed += 1; print(f'\u2705 Check 4: loaded=7, skipped=3')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: 7 rows in the database
    try:
        from sqlalchemy import select as sa_select
        db_count = len(session.execute(sa_select(Sale)).scalars().all())
        assert db_count == 7, \
            f'expected 7 rows in DB, got {db_count}'
        passed += 1; print(f'\u2705 Check 5: {db_count} Sale rows in DB')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def run_pipeline(csv_text: str, session) -> dict:
    raw_records  = extract(csv_text)
    valid        = [r for r in raw_records if validate_record(r)]
    transformed  = [transform_record(r) for r in valid]
    loaded_count = load(session, transformed)
    return {
        'extracted': len(raw_records),
        'loaded':    loaded_count,
        'skipped':   len(raw_records) - loaded_count,
    }
```

</details>